In [ ]:
# Installation of packages - Pinned versions (deterministic results)

%pip install --no-cache-dir \
    "IPython==9.11.0" \
    "datasets==5.0.1" \
    "matplotlib==3.10.9" \
    "numpy==2.4.3" \
    "pandas==2.3.3" \
    "scikit-learn==1.9.0" \
    "torch==2.13.0" \
    "tqdm==4.67.3" \
    "transformers==4.57.6" \
    "huggingface_hub==0.36.0" \
    "evaluate" \
    "accelerate" \
    "tokenizers" \
    "sentencepiece"

In [ ]:
# Loading and inspecting data
# Data is loaded into a pandas dataframe

import pandas as pd

# read data
df = pd.read_csv(
    "manual_codings.csv",
    encoding="utf-8-sig"
)

# dimensions
print("Shape:", df.shape)

# column names
print("\nColumns:")
print(df.columns.tolist())

# data types
print("\nDtypes:")
print(df.dtypes)


In [ ]:
# Note: The cell displays raw data (comments/posts). The output has been cleaned prior to sharing.
# The variable 'comment_text' contains the manually coded Facebook comments.
# Generate a new 'comment_text' variable called 'comment_text_clean' that has undergone text preprocessing.
# Minimal pre-processing: removes only URLs and redundant white space
# Examines the first 20 comments where this change has been made, to validate that the URLs have been removed correctly.

df["comment_text_clean"] = (
    df["comment_text"]
    .str.replace(r"https?://\S+|www\.\S+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

pd.set_option("display.max_colwidth", None)

changed = df[df["comment_text"] != df["comment_text_clean"]]
changed[["comment_text", "comment_text_clean"]].head(20)

In [ ]:
# Build dataset for negative stereotypes vs. no hate speech 
# Exclude rows where imp_nstereo == 0 and hs == 1 (meaning hate speech that is not coded as negative stereotypes)
n_dropped = ((df["imp_nstereo"] == 0) & (df["hs"] == 1)).sum()

df = df[~((df["imp_nstereo"] == 0) & (df["hs"] == 1))].copy()

print("Rows dropped (imp_nstereo==0 & hs==1):", n_dropped)
print("Total rows remaining:", df.shape[0])
print("\nDistribution of negative stereotyping in filtered dataset:")
print(df["imp_nstereo"].value_counts())

In [ ]:
# Determinism settings
# CUBLAS_WORKSPACE_CONFIG is set so that cuBLAS uses deterministic algorithms instead of freely choosing from several options.
# The seed is defined here and reused in the split cell and all training arguments below.
# cudnn.determinstic=True+cudnn.benchmark=False disables cuDNN's "select fastest kernel"-behavior, which can otherwise vary freely from run to run
# torch.use_deterministic_algorithms (True, warn_only=False) asks PyTorch to use deterministic implementations wherever possible
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "1512"

import random
import numpy as np
import torch

SEED = 1512

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
# Data split (train/dev/test)
# Split 80/10/10
# Stratified on negative stereotyping, meaning that the proportion of hate speech versus non-hate speech is kept roughly the same across all three parts.
# train is used to train all models
# dev is used for hyperparameter tuning (learning rate, batch size, epochs via early stopping).
# The test set is used only once in the end for the final evaluation of how well the model performs on entirely new comments that it has never seen before.

from sklearn.model_selection import train_test_split

# SEED already set above

# Set aside 10% as a final test set (stratified by negative stereotyping)
train_dev, test = train_test_split(
    df,
    test_size=0.10,
    random_state=SEED,
    stratify=df["imp_nstereo"],
)

# Split the remainder 90/10 into train/dev (= 10% of the total dataset), also stratified.
train, dev = train_test_split(
    train_dev,
    test_size=1 / 9,  # 10% of the original dataset, out of the remaining 90%
    random_state=SEED,
    stratify=train_dev["imp_nstereo"],
)

print("Train shape:", train.shape)
print("Dev shape:  ", dev.shape)
print("Test shape: ", test.shape)

In [ ]:
# Checking negative stereotyping distribution across train, dev, and test

vars_to_check = ["imp_nstereo"]

splits = {"train": train, "dev": dev, "test": test}

for var in vars_to_check:
    comparison = pd.concat(
        [splits[name][var].value_counts(normalize=True).mul(100).round(2).rename(f"{name}_%")
         for name in splits],
        axis=1,
    )
    print(f"\n--- {var} ---")
    print(comparison.fillna(0))

In [ ]:
# Converting to HuggingFace dataset
# Transformers expects data in its own dataset format (not in a pandas dataframe).
# The code selects only the two columns that the model actually needs (comment_text_clean and imp_nstereo).
# The 3 data splits are converted into train_ds, dev_ds and test_ds

from datasets import Dataset

train_df = train[["comment_text_clean", "imp_nstereo"]].dropna()
dev_df = dev[["comment_text_clean", "imp_nstereo"]].dropna()
test_df = test[["comment_text_clean", "imp_nstereo"]].dropna()

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
dev_ds = Dataset.from_pandas(dev_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

In [ ]:
# Model download – UCloud workaround, because UCloud is having trouble connecting to Hugging Face.
# Downloads the pre-trained DanskBERT model from HuggingFace and saves it locally in the ./DanskBERT folder. 
# Snapshot_download with allow_patterns is the Ucloud-specific solution that became necessary because Ucloud experienced issues with the standard connection to HuggingFace.
# Instead, only the specific files required by the model (weights, config, and tokenizer) are downloaded.

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="vesteinn/DanskBERT",
    local_dir="./DanskBERT",
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "sentencepiece.bpe.model",
    ],
)

In [ ]:
# Loading tokenizer

from transformers import AutoTokenizer

model_name = "./DanskBERT"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True,
)

In [ ]:
# Text length check
# Investigating the typical word count of the comments, in order to make an informed decision regarding the maximum allowed length of the text before it gets cut off (truncated) during tokenization.

df["len"] = df["comment_text_clean"].str.split().str.len()
print(df["len"].describe())
print("\n95. percentil (ord):", df["len"].quantile(0.95))
# The 95th percentile of the comments contains approximately 164 words, which is why 256 tokens makes sense as max-length

In [ ]:
# Tokenization
# Each comment is processed through the tokenizer with `truncation=True` and `max_length=256`—meaning that comments exceeding 256 tokens are truncated (based on the fact that this covers the vast majority of comments).
# padding=False means that we do not pad short comments to the same length here; this happens dynamically during the actual training, which saves computation time.

def tokenize(batch):
    return tokenizer(
        batch["comment_text_clean"],
        truncation=True,
        max_length=256,
        padding=False,  # dynamic padding is handled by DataCollatorWithPadding
    )

train_ds = train_ds.map(tokenize, batched=True)
dev_ds = dev_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
# Formatting to PyTorch

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_ds = train_ds.rename_column("imp_nstereo", "labels")
dev_ds = dev_ds.rename_column("imp_nstereo", "labels")
test_ds = test_ds.rename_column("imp_nstereo", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
dev_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Model function
# Defines model_init() – a function that (re)creates a fresh DanskBERT model with a classification head for 2 classes (imp_nstereo=0 or 1).

from transformers import AutoModelForSequenceClassification

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        "./DanskBERT",
        num_labels=2,
        local_files_only=True,
        attn_implementation="eager",
    )

In [ ]:
# Metrics
# macro_f1 is the one that is subsequently used to determine the best hyperparameter combination.

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )
    weighted_f1 = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )[2]
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "precision_macro": precision,
        "recall_macro": recall,
    }

In [ ]:
# Hyperparameter grid search
# For each combination of learning rate (5 values) and batch size (2 values) 
# 10 combinations in total – a fresh model is trained for up to 6 epochs, but with automatic stopping (EarlyStoppingCallback) if the performance on dev_ds does not improve for 2 consecutive epochs.
# Collected in results_df, sorted with the highest macro F1 at the top.

from itertools import product
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from tqdm.auto import tqdm
import gc
import time
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

learning_rates = [1e-5, 2e-5, 3e-5, 4e-5, 5e-5]
batch_sizes = [16, 32]
MAX_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 2

def best_epoch_and_score(trainer, metric_key="eval_macro_f1"):
    """Find the epoch (and score) at which the tracked metric peaked during a run,
    by scanning Trainer's own eval log history."""
    best_epoch, best_score = None, -1
    for entry in trainer.state.log_history:
        if metric_key in entry and entry[metric_key] > best_score:
            best_score = entry[metric_key]
            best_epoch = entry.get("epoch")
    return best_epoch, best_score

results = []
configs = list(product(learning_rates, batch_sizes))
best_so_far = -1.0

outer_bar = tqdm(configs, desc="Grid search", unit="config")

for lr, batch_size in outer_bar:
    config_start = time.time()
    outer_bar.set_postfix({
        "lr": lr,
        "batch": batch_size,
        "best_macro_f1_so_far": f"{best_so_far:.4f}" if best_so_far >= 0 else "n/a",
    })
    print(f"\nTraining: lr={lr}, batch_size={batch_size}, up to {MAX_EPOCHS} epochs (early stopping)")

    training_args = TrainingArguments(
            output_dir=f"./bert_hs_grid/lr_{lr}_batch_{batch_size}",
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=MAX_EPOCHS,
            weight_decay=0.01,
            warmup_ratio=0.06,
            load_best_model_at_end=True,
            metric_for_best_model="eval_macro_f1",
            greater_is_better=True,
            seed=SEED,
            data_seed=SEED,
            logging_dir="./logs",
            logging_strategy="steps",
            logging_steps=25,          
            disable_tqdm=False,        
            fp16=False,
            report_to="none",
            dataloader_num_workers=0,
        )

    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer.train()
    eval_result = trainer.evaluate()
    epoch, _ = best_epoch_and_score(trainer)

    results.append({
        "learning_rate": lr,
        "batch_size": batch_size,
        "best_epoch": round(epoch) if epoch is not None else MAX_EPOCHS,
        "accuracy": eval_result["eval_accuracy"],
        "macro_f1": eval_result["eval_macro_f1"],
        "weighted_f1": eval_result["eval_weighted_f1"],
        "precision_macro": eval_result["eval_precision_macro"],
        "recall_macro": eval_result["eval_recall_macro"],
    })

    best_so_far = max(best_so_far, eval_result["eval_macro_f1"])

    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Config finished in {(time.time() - config_start) / 60:.1f} min "
          f"(macro_f1={eval_result['eval_macro_f1']:.4f})")

outer_bar.close()

results_df = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)

print(results_df.to_string(index=False))

In [ ]:
# Displaying result table

results_df

In [ ]:
# Final model and one evaluation on the test set
# Retrieves the best combination (top row in results_df) and retrains a single model using exactly those hyperparameters.
# This model is then evaluated—for the first and only time—on test_ds.
# The result is printed as numbers and visualized as a confusion matrix.

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best = results_df.iloc[0]
best_lr = float(best["learning_rate"])
best_batch_size = int(best["batch_size"])
best_epochs = int(best["best_epoch"])

print(f"Best configuration (chosen from DEV): lr={best_lr}, batch_size={best_batch_size}, epochs={best_epochs}")

best_args = TrainingArguments(
    output_dir="./bert_hs_best",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=best_lr,
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    num_train_epochs=best_epochs,
    weight_decay=0.01,
    warmup_ratio=0.06,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    dataloader_num_workers=0,  
)

best_trainer = Trainer(
    model_init=model_init,
    args=best_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds, 
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

best_trainer.train()

best_trainer.save_model("./final_model_imp_nstereo")
tokenizer.save_pretrained("./final_model_imp_nstereo")

predictions = best_trainer.predict(test_ds)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

test_metrics = compute_metrics((predictions.predictions, y_true))
print("\nFinal test metrics (unpartial, test set used first and only time):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)

display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix – Danish BERT: nstereo/no-hs (test)")
plt.savefig("confusion_matrix_danskbert.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Printing package versions 
import sys
import importlib.metadata as metadata

def print_package_versions():
    imported_modules = set()
    for name, val in list(globals().items()):
        if hasattr(val, "__name__"):
            imported_modules.add(val.__name__.split(".")[0])
        elif hasattr(val, "__module__") and val.__module__:
            imported_modules.add(val.__module__.split(".")[0])

    installed = {dist.metadata["Name"].lower(): dist.version 
                 for dist in metadata.distributions()}
    
    # Known mismatches between import name and pip package name
    aliases = {
        "sklearn": "scikit-learn", "PIL": "Pillow", "cv2": "opencv-python",
        "bs4": "beautifulsoup4", "yaml": "PyYAML",
    }

    results = {}
    for mod in sorted(imported_modules):
        pkg_name = aliases.get(mod, mod)
        version = installed.get(pkg_name.lower())
        if version:
            results[pkg_name] = version

    for pkg, ver in sorted(results.items()):
        print(f"{pkg}=={ver}")

print(f"Python: {sys.version.split()[0]}\n")
print_package_versions()